In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from river import drift, linear_model, metrics, optim

In [ ]:
df = pd.read_csv('../data/LSudden_3.csv')
X_col = df.columns[:5]
y_col = df.columns[5]
X = df[X_col]
y = df[y_col]

In [ ]:
df.describe()

In [ ]:
def learn_many(model, df):
    for _, row in df.iterrows():
        X_i = dict(row[X_col])
        y_i = row[y_col]
        model.learn_one(X_i, y_i)

In [ ]:
train_size = 1000
model = linear_model.SoftmaxRegression()
learn_many(model, df[:train_size])

In [ ]:
adwin = drift.ADWIN()
accuracy = metrics.Accuracy()
accuracies = []
opt = optim.losses.CrossEntropy()
detection_points = []

for i, row in df[train_size:].iterrows():
    X_i = row[X_col]
    y_i = row[y_col]

    y_pred_proba = model.predict_proba_one(X_i)
    loss = opt(y_i, y_pred_proba)
    adwin.update(loss)

    y_pred = model.predict_one(X_i)
    accuracy.update(y_i, y_pred)
    accuracies.append(accuracy.get())

    if adwin.drift_detected:
        print(f'Drift detected @ sample #{i}')
        detection_points.append(i)
        print('Retraining model')
        start_idx = max(0, i - train_size + 1)
        learn_many(model, df[start_idx: i + 1])

The output of the first experiment I made:

```
Drift detected @ sample #200167
Retraining model
Drift detected @ sample #400103
Retraining model
Drift detected @ sample #538087
Retraining model
Drift detected @ sample #600743
Retraining model
Drift detected @ sample #800071
Retraining model
```

The documentation of the dataset stated that an abrupt drift occurs every 200k points.
This suggests that even this very simple model did quite well thanks to ADWIN.

In [ ]:
# Expected drift points based on documentation (every 200k samples)
expected_drift_points = [200000, 400000, 600000, 800000]

# Compute reaction time for each expected drift and track matched detections
reaction_times = []
matched_detections = []

for expected in expected_drift_points:
    # Find the closest detection point after the expected drift
    detected_after = [d for d in detection_points if d >=
                      expected and d not in matched_detections]
    if detected_after:
        detected = detected_after[0]
        matched_detections.append(detected)
        reaction_time = detected - expected
        reaction_times.append(reaction_time)
        print(
            f'Expected drift @ {expected}, Detected @ {detected}, Reaction time: {reaction_time} samples')
    else:
        print(f'Expected drift @ {expected}, no detection found')

if reaction_times:
    print(
        f'Mean reaction time: {sum(reaction_times) / len(reaction_times):.0f} samples')
    print(f'Max reaction time: {max(reaction_times)} samples')
    print(f'Min reaction time: {min(reaction_times)} samples')
else:
    print('No reaction times available')

# False positives: detections that were not matched to expected drifts
false_positives = [d for d in detection_points if d not in matched_detections]
print(f'False positive detections ({len(false_positives)}): {false_positives}')

The output of the above code was as follows:

```
Expected drift @ 200000, Detected @ 200167, Reaction time: 167 samples
Expected drift @ 400000, Detected @ 400103, Reaction time: 103 samples
Expected drift @ 600000, Detected @ 600743, Reaction time: 743 samples
Expected drift @ 800000, Detected @ 800071, Reaction time: 71 samples
Mean reaction time: 271 samples
Max reaction time: 743 samples
Min reaction time: 71 samples
False positive detections (1): [538087]
```

In [ ]:
plt.plot(accuracies)
plt.title('Accuracy')
plt.show()